# Helion pipeline walkthrough

This notebook reads the completed research run. It does not retrain, alter sources or turn simulated reports into observed data. The accompanying `benchmark_report.md` is the complete results report.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image
RUN = Path.cwd()
if not (RUN / "run_manifest.json").exists():
    candidates = list(RUN.glob("artifacts/helion_pipeline/*/run_manifest.json"))
    if len(candidates) != 1:
        raise RuntimeError("Open this notebook with its containing run directory as the working directory")
    RUN = candidates[0].parent
manifest = json.loads((RUN / "run_manifest.json").read_text())
print(manifest["version"], "| synthetic research | not production qualified")

## 1. What the model learns

Each rejected stack has seven binary targets; coexisting faults are allowed. Probabilities describe fault hypotheses, not the best test or the probability of completing an investigation. Several acceptance predictors contain generator shortcuts, so compare full-context and manufacturing-only results.

In [ ]:
metrics = pd.read_csv(RUN / "predictive_metrics.csv")
display(metrics[(metrics.split == "test") & (metrics.mechanism == "macro")])
display(Image(filename=str(RUN / "figures/prediction_comparison.png")))

## 2. Calibration and limited evidence

The test partition has six die-crack cases and two cases with coexisting faults. Binned curves and bootstrap intervals describe this synthetic sample; they do not qualify future fab use.

In [ ]:
display(Image(filename=str(RUN / "figures/calibration.png")))
display(pd.read_csv(RUN / "predictive_paired_comparisons.csv"))

## 3. From predictions to eligible procedures

The evidence engine applies shared scope, concern-opening, co-fault and destruction rules before selection. The candidate rules and CT-first compatibility arm use the same concern-priority order; the model heuristic can reorder eligible concern work. The heuristic values positive unresolved coverage per dollar, including the assumed gross-delamination fraction. It is not an optimal planner; negative evidence and bounded repeats after inconclusive nondestructive attempts still matter for closure.

In [ ]:
walkthroughs = json.loads((RUN / "walkthroughs.json").read_text())
rows = [{"case": name, **{k: item["summary"][k] for k in ["spent_cost", "pending_cost", "complete", "unresolved_count"]}} for name, item in walkthroughs.items() if "summary" in item]
display(pd.DataFrame(rows))
for name in ["B", "C", "C_inconclusive"]:
    print(name, walkthroughs[name]["boundary"])
    display(pd.DataFrame([{ "procedure": e["procedure"], "cumulative_cost": e["cumulative_cost"], "unresolved": ", ".join(e["unresolved_mechanisms"])} for e in walkthroughs[name]["events"]]))

## 4. Compare entire bounded investigations

Every started case stays in the denominator. A cheaper partial record is not cost-to-completion savings. The columns separate consumed and pending costs, diagnostic errors and unresolved mechanisms; manual-review continuation costs are unknown.

In [ ]:
decisions = pd.read_csv(RUN / "decision_metrics.csv")
display(decisions[(decisions.scenario == "base") & (decisions.split == "test")])
display(Image(filename=str(RUN / "figures/diagnostic_tradeoffs.png")))

## 5. Scheduling changes elapsed time independently of fault probability

The calendar ends after 24 hours. Case D compares A-first and B-first CT bookings with actual staff phases. Both consume $240; the B-first ordering meets the two scoped milestones if reports are conclusive. No future SEM slot is invented.

In [ ]:
display(Image(filename=str(RUN / "figures/scheduling.png")))
scheduling = json.loads((RUN / "scheduling.json").read_text())
display(scheduling["base"]["B"])

## 6. Uncertainty and stress tests

Lot-cluster bootstrap uncertainty resamples cases by lot. Monte Carlo variation changes hypothetical reports while holding cases fixed. Rate, outcome and outage stresses change assumptions and are not confidence intervals. Correlated errors and narrower scope remain qualitative limitations.

In [ ]:
display(pd.read_csv(RUN / "decision_uncertainty.csv").query("split == 'test' and metric in ['cost', 'complete', 'missed_faults']"))
display(decisions[(decisions.scenario != "base") & (decisions.split == "test")][["scenario", "arm", "cost", "complete", "missed_faults"]])

## 7. What would change the operational recommendation?

Actual SOP and repeat decisions, independent complete audits, validated procedure performance, qualified multi-day calendars and prospective comparative evidence. Neither retrospective classifier accuracy nor this synthetic replay establishes production readiness. Engineer closure and reviewed model release remain necessary.